<a href="https://colab.research.google.com/github/taha-qureshi127/Medical-chatbot/blob/main/medical%20chatbot%20task%20arch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225

In [3]:
from huggingface_hub import login
import wandb
from google.colab import userdata # Use Colab's secrets module

# Retrieve tokens from Colab's secret manager
hf_token = userdata.get('HUGGINGFACE_TOKEN')
wb_token = userdata.get('wandb')

# Log into the platforms
login(hf_token)
wandb.login(key=wb_token)
run = wandb.init(project='Fine-tune-DeepSeek-R1-Medical', job_type="training", anonymous="allow")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: khalidtaha515 (khalidtaha515-cust) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


In [4]:
from unsloth import FastLanguageModel

max_seq_length = 2048

# This downloads the model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True, # This enables quantization to save GPU memory
    token = hf_token,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [5]:
from datasets import load_dataset

train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    inputs       = examples["Question"]
    cots         = examples["Complex_CoT"]
    outputs      = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split = "train[0:500]")
dataset = dataset.map(formatting_prompts_func, batched = True)

print("Dataset loaded and formatted successfully!")

README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json: reconstructing file:   0%|          |  0.00B / 58.2MB            

medical_o1_sft.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset loaded and formatted successfully!


In [6]:

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # 0 is optimized for speed
    bias = "none",    # "none" is optimized for speed
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of memory
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapters successfully attached to the model!")

Unsloth 2026.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters successfully attached to the model!


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # How many examples it looks at once
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Total number of training steps for this test run
        learning_rate = 2e-4, # How fast the model learns
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit", # A memory-efficient optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

# 2. Start the training!
print("Starting the training process...")
trainer_stats = trainer.train()
print("Training complete!")

Starting the training process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,0.989003
20,1.059308
30,1.290575
40,1.443069
50,1.641171
60,1.740889


Training complete!


In [11]:
import textwrap

FastLanguageModel.for_inference(model)

prompt = train_prompt_style.format(
    "A 61-year-old woman complains of involuntary urine loss when coughing or sneezing. What is the most likely diagnosis?",
    "",
    ""
)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=1200, use_cache=True)

input_length = inputs.input_ids.shape[1]
generated_tokens = outputs[0][input_length:]

# This line cleans up the strange symbols
response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
cleaned_text = response_text.replace("Ġ", " ").replace("Ċ", "\n")

print("--- AI RESPONSE ---\n")
for line in cleaned_text.split("\n"):
    if line.strip():
        print(textwrap.fill(line, width=80))
    else:
        print()

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- AI RESPONSE ---


Okay, so I'm trying to figure out the most likely diagnosis for a 61-year-old
woman who's experiencing involuntary urine loss when she coughs or sneezes. Let
me break this down step by step.

First, I remember that when a person loses urine involuntarily during activities
like coughing or sneezing, it's often related to something affecting their
bladder or the nerves controlling it. The most common condition that comes to
mind is called "Urinary Incontinence." But there are different types of
incontinence, so I need to narrow it down.

The woman is 61, which is over 60, so age might be a factor. I think
incontinence can happen due to aging, where the bladder muscle weakens or the
nerves don't work as well. That could lead to what's called "stress
incontinence." I think stress incontinence is when you lose urine when you
cough, sneeze, or do other movements that put pressure on the bladder.

Wait, but there's also something called "urgency of urination" or "frequen